**German Credit Dataset**

# 01 - Exploratory Data Analysis

**Objectives**
- Understand the context and the potential of the dataset
- Assess data quality, including missing values, outliers, and undocumented categories
- Explore the relationship between clients’ personal characteristics and the probability of default
- Explore the relationship between clients’ payment history and the probability of default

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, roc_curve
from sklearn.metrics import confusion_matrix, precision_recall_curve, f1_score, precision_score, recall_score
import re

## 1. Load Data

**Source:** https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data

In [ ]:
file_path = '../data/raw/german.data'
df = pd.read_csv(file_path, sep=r'\s+', header=None)

## 2. EDA

### 2.1. Dataset Description

In [ ]:
print(f"Dataset shape: {df.shape}")
print(f"Dataset columns: {df.columns}")

In [ ]:
df.info()

In [ ]:
df.head(20)

Identifying potential categorical features:

In [ ]:
summary = []

for col in df.columns:
    uniques = df[col].unique()
    n_unique = len(uniques)

    if n_unique > 5:
        example_values = list(uniques[:5]) + ["..."]
    else:
        example_values = list(uniques)

    summary.append({
        "column": col,
        "dtype": df[col].dtype,
        "unique_values": n_unique,
        "example_values": example_values
    })

summary_df = pd.DataFrame(summary)
print(summary_df)

#### Dataset's descriptive characteristics

The following description is based on the information provided in the original source where this dataset is published. The dataset corresponds to a credit risk classification problem (commonly known as the *German Credit Dataset*) and contains 20 input attributes and 1 target variable.

- 1,000 rows
- 21 columns (including the target column)
- No missing values

**Monetary values are expressed in DM (Deutsche Mark)*


| Variable Name | Role   | Type        | Description                                              | Units  |
|---------------|--------|-------------|----------------------------------------------------------|--------|
| Attribute1    | Feature| Categorical | Status of existing checking account                      |        |
| Attribute2    | Feature| Integer     | Duration                                                 | months |
| Attribute3    | Feature| Categorical | Credit history                                           |        |
| Attribute4    | Feature| Categorical | Purpose                                                  |        |
| Attribute5    | Feature| Integer     | Credit amount                                            |        |
| Attribute6    | Feature| Categorical | Savings account / bonds                                  |        |
| Attribute7    | Feature| Categorical | Present employment since                                 |        |
| Attribute8    | Feature| Integer     | Installment rate in percentage of disposable income      |        |
| Attribute9    | Feature| Categorical | Personal status and sex                                  |        |
| Attribute10   | Feature| Categorical | Other debtors / guarantors                               |        |
| Attribute11   | Feature| Integer     | Present residence since                                  |        |
| Attribute12   | Feature| Categorical | Property                                                 |        |
| Attribute13   | Feature| Integer     | Age                                                      | years  |
| Attribute14   | Feature| Categorical | Other installment plans                                  |        |
| Attribute15   | Feature| Categorical | Housing                                                  |        |
| Attribute16   | Feature| Integer     | Number of existing credits at this bank                  |        |
| Attribute17   | Feature| Categorical | Job                                                      |        |
| Attribute18   | Feature| Integer     | Number of people being liable to provide maintenance for |        |
| Attribute19   | Feature| Binary      | Telephone                                                |        |
| Attribute20   | Feature| Binary      | Foreign worker                                           |        |
| class         | Target | Binary      | 1 = Good, 2 = Bad                                        |        |

**Attribute 1 — Status of Existing Checking Account (Categorical)**

- A11: balance < 0 DM  
- A12: 0 ≤ balance < 200 DM  
- A13: balance ≥ 200 DM or salary assigned for at least 1 year  
- A14: no checking account  


**Attribute 2 — Duration (Numerical)**

- Duration of the credit in months.


**Attribute 3 — Credit History (Categorical)**

- A30: no credits taken / all credits paid back duly  
- A31: all credits at this bank paid back duly  
- A32: existing credits paid back duly until now  
- A33: delay in paying off in the past  
- A34: critical account / other credits existing (not at this bank)  


**Attribute 4 — Purpose (Categorical)**

- A40: car (new)  
- A41: car (used)  
- A42: furniture / equipment  
- A43: radio / television  
- A44: domestic appliances  
- A45: repairs  
- A46: education  
- A47: vacation (not used in practice)  
- A48: retraining  
- A49: business  
- A410: others  



**Attribute 5 — Credit Amount (Numerical)**

- Amount of credit requested (in DM).



**Attribute 6 — Savings Account / Bonds (Categorical)**

- A61: balance < 100 DM  
- A62: 100 ≤ balance < 500 DM  
- A63: 500 ≤ balance < 1000 DM  
- A64: balance ≥ 1000 DM  
- A65: unknown / no savings account  



**Attribute 7 — Present Employment Since (Categorical)**

- A71: unemployed  
- A72: < 1 year  
- A73: 1 to 4 years  
- A74: 4 to 7 years  
- A75: ≥ 7 years  



**Attribute 8 — Installment Rate (Numerical)**

- Installment rate as a percentage of disposable income.

**This attribute indicates the proportion of the applicant’s disposable income used to pay the loan installment. Although described as numerical in the original source, it only takes the discrete values 1, 2, 3, and 4, representing increasing burden levels, and therefore behaves as an ordinal categorical variable in practice.*



**Attribute 9 — Personal Status and Sex (Categorical)**

- A91: male, divorced/separated  
- A92: female, divorced/separated/married  
- A93: male, single  
- A94: male, married/widowed  
- A95: female, single  



**Attribute 10 — Other Debtors / Guarantors (Categorical)**

- A101: none  
- A102: co-applicant  
- A103: guarantor 

**This attribute indicates whether the applicant shares the responsibility for the loan with another person. It can represent no additional support, a co-applicant who jointly repays the loan, or a guarantor who agrees to repay the debt if the applicant defaults.* 



**Attribute 11 — Present Residence Since (Numerical)**

- Number of years at the current residence.

**Although described as numerical in the original source, this attribute only takes the discrete values 1, 2, 3, and 4, and therefore behaves as an ordinal categorical variable in practice.*



**Attribute 12 — Property (Categorical)**

- A121: real estate  
- A122: building society savings agreement / life insurance  
- A123: car or other property (not in Attribute 6)  
- A124: unknown / no property  


**Attribute 13 — Age (Numerical)**

- Age in years.


**Attribute 14 — Other Installment Plans (Categorical)**

- A141: bank  
- A142: stores  
- A143: none  

**This attribute indicates whether the applicant already has other active installment-based debts*



**Attribute 15 — Housing (Categorical)**

- A151: rent  
- A152: own  
- A153: for free  



**Attribute 16 — Number of Existing Credits at This Bank (Numerical)**

- Total number of credits currently held at this bank.

**Although this attribute is described as numerical in the original source, it only takes the discrete values 1, 2, 3, and 4, and therefore behaves as an ordinal categorical variable in practice.*



**Attribute 17 — Job (Categorical)**

- A171: unemployed / unskilled (non-resident)  
- A172: unskilled (resident)  
- A173: skilled employee / official  
- A174: management / self-employed / highly qualified employee / officer  



**Attribute 18 — Number of People Liable for Maintenance (Numerical)**

- Number of dependents financially supported by the applicant.

**Although this attribute is described as numerical in the original source, it only takes the discrete values 1 and 2, and therefore behaves as a categorical variable in practice.*



**Attribute 19 — Telephone (Categorical)**

- A191: none  
- A192: yes, registered under the customer’s name  



**Attribute 20 — Foreign Worker (Categorical)**

- A201: yes  
- A202: no  



**Target Variable — Credit Risk**

The target variable indicates the credit risk of the applicant:

- 1: Good credit risk  
- 2: Bad credit risk  

For convenience and standardization purposes, a copy of the original dataset is created below with renamed fields. Additionally, a mapping of categorical data labels is created and organized by feature name, in order to facilitate the plotting of charts with meaningful labels:

In [ ]:
df_standard = df.copy()

In [ ]:
# renaming columns
df_standard = df_standard.rename(columns={
    0:  "checking_account_status",
    1:  "duration_months",
    2:  "credit_history",
    3:  "purpose",
    4:  "credit_amount",
    5:  "savings_account_status",
    6:  "employment_status",
    7:  "installment_rate",
    8:  "marriage_status_sex",
    9:  "guarantors",
    10: "residence_duration",
    11: "property",
    12: "age",
    13: "other_debts",
    14: "housing",
    15: "existing_credits_count",
    16: "job",
    17: "dependents",
    18: "own_telephone?",
    19: "foreign_worker?",
    20: "good_client?"   # target variable
})

In [ ]:
# Mapping categorical variables to integers
category_map_numeric = {
    "checking_account_status": {
        "A11": 1,
        "A12": 2,
        "A13": 3,
        "A14": 4
    },

    "credit_history": {
        "A30": 1,
        "A31": 2,
        "A32": 3,
        "A33": 4,
        "A34": 5
    },

    "purpose": {
        "A40": 1,
        "A41": 2,
        "A42": 3,
        "A43": 4,
        "A44": 5,
        "A45": 6,
        "A46": 7,
        "A47": 8,
        "A48": 9,
        "A49": 10,
        "A410": 11
    },

    "savings_account_status": {
        "A61": 1,
        "A62": 2,
        "A63": 3,
        "A64": 4,
        "A65": 5
    },

    "employment_status": {
        "A71": 1,
        "A72": 2,
        "A73": 3,
        "A74": 4,
        "A75": 5
    },

    "marriage_status_sex": {
        "A91": 1,
        "A92": 2,
        "A93": 3,
        "A94": 4,
        "A95": 5
    },

    "guarantors": {
        "A101": 1,
        "A102": 2,
        "A103": 3
    },

    "property": {
        "A121": 1,
        "A122": 2,
        "A123": 3,
        "A124": 4
    },

    "other_debts": {
        "A141": 1,
        "A142": 2,
        "A143": 3
    },

    "housing": {
        "A151": 1,
        "A152": 2,
        "A153": 3
    },

    "job": {
        "A171": 1,
        "A172": 2,
        "A173": 3,
        "A174": 4
    },

    "own_telephone?": {
        "A191": 0,
        "A192": 1
    },

    "foreign_worker?": {
        "A201": 1,
        "A202": 0
    },

    "good_client?": {
        1: 1,
        2: 0
    },
}

for col, mapping in category_map_numeric.items():
    df_standard[col] = df_standard[col].map(mapping)

In [ ]:
# mapping categorical data labels
category_map = {
    "checking_account_status": {
        1: "< 0 DM",
        2: "0 to < 200 DM",
        3: ">= 200 DM or salary >= 1 year",
        4: "no checking account"
    },

    "credit_history": {
        1: "no credits / all paid duly",
        2: "all credits paid duly",
        3: "existing credits paid duly",
        4: "past payment delays",
        5: "critical account / other credits"
    },

    "purpose": {
        1: "new car",
        2: "used car",
        3: "furniture/equipment",
        4: "radio/TV",
        5: "domestic appliances",
        6: "repairs",
        7: "education",
        8: "vacation",
        9: "retraining",
        10: "business",
        11: "other"
    },

    "savings_account_status": {
        1: "< 100 DM",
        2: "100 to < 500 DM",
        3: "500 to < 1000 DM",
        4: ">= 1000 DM",
        5: "unknown / none"
    },

    "employment_status": {
        1: "unemployed",
        2: "< 1 year",
        3: "1 to 4 years",
        4: "4 to 7 years",
        5: ">= 7 years"
    },

    "marriage_status_sex": {
        1: "male divorced/separated",
        2: "female divorced/separated/married",
        3: "male single",
        4: "male married/widowed",
        5: "female single"
    },

    "guarantors": {
        1: "none",
        2: "co-applicant",
        3: "guarantor"
    },

    "property": {
        1: "real estate",
        2: "savings/life insurance",
        3: "car or other property",
        4: "unknown/none"
    },

    "other_debts": {
        1: "bank",
        2: "stores",
        3: "none"
    },

    "housing": {
        1: "rent",
        2: "own",
        3: "free"
    },

    "job": {
        1: "unemployed/unskilled (non-resident)",
        2: "unskilled (resident)",
        3: "skilled employee",
        4: "management/self-employed / highly qualified"
    },

    "own_telephone?": {
        1: "yes",
        0: "no"
    },

    "foreign_worker?": {
        1: "yes",
        0: "no"
    },

    "good_client?": {
        1: "yes",
        0: "no"
    }
}

In [ ]:
df_standard.head(20)

In [ ]:
# Numerical columns
numerical_cols = ['duration_months', 'credit_amount', 'installment_rate', 'residence_duration', 'age', 'existing_credits_count', 'dependents']

# Categorical and binary columns (excluding target variable)
categorical_cols = ['checking_account_status', 'credit_history', 'purpose', 'savings_account_status', 'employment_status', 'marriage_status_sex', 'guarantors', 'property', 'other_debts', 'housing', 'job', 'own_telephone?', 'foreign_worker?']

### 2.3. Exploratory Visualizations

#### 2.3.1. Data Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

df_standard['good_client?'].value_counts().sort_index().plot(kind='bar', ax=ax)

ax.set_title('Distribution of good and bad clients')
ax.set_xlabel('Client classification')
ax.set_ylabel('Count')

ax.set_xticklabels(['Bad', 'Good'], rotation=0)

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()
plt.show()

From the analysis of the chart above, it is evident that the data are imbalanced (as expected), with more good clients than bad clients.

This imbalance can have fairness implications, as predictive models trained on such data may become biased toward the majority group, leading to systematically worse performance for minority groups (e.g., bad payers).

In [ ]:
# Plotting distribution of categorical variables
for col in categorical_cols:
    fig, ax = plt.subplots(figsize=(10, 6))

    mapped_values = df_standard[col].map(category_map[col])
    value_counts = mapped_values.value_counts().sort_index()

    if value_counts.empty:
        ax.text(0.5, 0.5, f'No mapped values for {col}', ha='center', va='center')
        ax.set_axis_off()
        plt.tight_layout()
        plt.show()
        continue

    value_counts.plot(kind='bar', ax=ax, color='steelblue')

    ax.set_title(f'Distribution of {col.replace("_", " ").title()}', fontsize=14, weight='bold')
    ax.set_xlabel(col.replace("_", " ").title())
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

    for container in ax.containers:
        ax.bar_label(container, padding=3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Plotting distribution of numerical variables
for col in numerical_cols:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(df_standard[col], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].set_title(f'Distribution of {col.replace("_", " ").title()}', fontsize=12, weight='bold')
    axes[0].set_xlabel(col.replace("_", " ").title())
    axes[0].set_ylabel('Frequency')
    axes[0].grid(axis='y', linestyle='--', alpha=0.3)
    
    # Boxplot
    axes[1].boxplot(df_standard[col], vert=False, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
    axes[1].set_title(f'Boxplot of {col.replace("_", " ").title()}', fontsize=12, weight='bold')
    axes[1].set_xlabel(col.replace("_", " ").title())
    axes[1].grid(axis='x', linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In the plots “Distribution of Installment Rate”, “Distribution of Residence Duration”, “Distribution of Existing Credits Count”, and “Distribution of Dependents”, it is again possible to observe the categorical-like behavior of these variables, even though they are described as numerical. Although the values themselves are meaningful within the context of each feature, in real-world scenarios it is extremely unlikely that, in a sample of 1,000 clients, all individuals would have only 1 or 2 dependents (and never 0 or 3), for example.

#### 2.3.2. Bad Client Rate Analysis Across Features

In [ ]:
# Analyzing relationship between categorical variables and bad client rate
for col in categorical_cols:
    viz_data = df_standard.copy()
    
    viz_data[f'{col}_label'] = viz_data[col].map(category_map[col])
    
    bad_client_rate = viz_data.groupby(f'{col}_label')['good_client?'].apply(lambda x: (1 - x).mean())
    
    x_order = list(category_map[col].values())
    
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    sns.countplot(
        data=viz_data, 
        x=f'{col}_label', 
        order=x_order,
        ax=ax1, 
        palette='Blues', 
        hue=f'{col}_label', 
        legend=False
    )
    ax1.set_ylabel('Count of Clients')
    ax1.set_xlabel(col.replace("_", " ").title())
    ax1.tick_params(axis='x', rotation=45)
    
    ax2 = ax1.twinx()
    y_values = bad_client_rate.reindex(x_order)
    
    ax2.plot(range(len(x_order)), y_values, color='r', marker='o', linestyle='-', linewidth=2)
    ax2.set_ylabel('Bad Client Rate', color='r')
    ax2.tick_params(axis='y', labelcolor='r')
    ax2.set_ylim(0, max(y_values.dropna()) * 1.2 if max(y_values.dropna()) > 0 else 0.5)

    for i, v in enumerate(y_values):
        if pd.notnull(v):
            ax2.text(i, v, f'{v:.1%}', color='r', ha='center', va='bottom', fontweight='bold')
    
    plt.title(f'Client Count and Bad Client Rate by {col.replace("_", " ").title()}')
    plt.tight_layout()
    plt.show()

In [ ]:
# Relationship between numerical variables and bad client rate 
for col in numerical_cols:
    viz_data = df_standard.copy()
    s = viz_data[col]
    n_unique = s.nunique(dropna=True)

    # Binning strategy:
    # - Few distinct values (≤10): use the values themselves as categories (ascending order)
    # - Many distinct values: use quantiles (qcut). If it fails, use equal cuts (cut).
    if n_unique <= 10:
        viz_data[f'{col}_bin'] = pd.Categorical(s, ordered=True)
        x_order = sorted(s.dropna().unique())
    else:
        try:
            viz_data[f'{col}_bin'] = pd.qcut(s, q=6, duplicates='drop')
        except Exception:
            viz_data[f'{col}_bin'] = pd.cut(s, bins=6, include_lowest=True)
        x_order = list(viz_data[f'{col}_bin'].cat.categories)

    bad_rate = viz_data.groupby(f'{col}_bin', observed=False)['good_client?'].apply(lambda x: (1 - x).mean())

    fig, ax1 = plt.subplots(figsize=(12, 6))
    sns.countplot(
        data=viz_data,
        x=f'{col}_bin',
        order=x_order,
        ax=ax1,
        palette='Blues',
        hue=f'{col}_bin',
        legend=False
    )
    ax1.set_ylabel('Count of Clients')
    ax1.set_xlabel(col.replace("_", " ").title())
    ax1.tick_params(axis='x', rotation=45)

    ax2 = ax1.twinx()
    y_values = bad_rate.reindex(x_order)
    ax2.plot(range(len(x_order)), y_values, color='r', marker='o', linestyle='-', linewidth=2)
    ax2.set_ylabel('Bad Client Rate', color='r')
    ax2.tick_params(axis='y', labelcolor='r')
    ymax = y_values.dropna().max() if y_values.dropna().size > 0 else 0
    ax2.set_ylim(0, max(ymax * 1.2, 0.5))

    for i, v in enumerate(y_values):
        if pd.notnull(v):
            ax2.text(i, v, f'{v:.1%}', color='r', ha='center', va='bottom', fontweight='bold')

    plt.title(f'Client Count and Bad Client Rate by {col.replace("_", " ").title()}')
    plt.tight_layout()
    plt.show()

### 2.4. Correlation Analysis

In [ ]:
# Calculate correlation matrix for numerical variables
correlation_matrix = df_standard[numerical_cols + ['good_client?']].corr()

fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(
    correlation_matrix,
    annot=True,  
    fmt='.2f',   
    cmap='coolwarm', 
    center=0,    
    square=True, 
    linewidths=0.5,  
    cbar_kws={'label': 'Correlation Coefficient'},
    ax=ax
)

ax.set_title('Correlation Matrix: Numerical Features and Target Variable', fontsize=14, weight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
target_corr = correlation_matrix['good_client?'].sort_values(ascending=False)

target_corr_abs = target_corr.abs().sort_values(ascending=False)

# Remove the target variable itself (self-correlation)
target_corr_abs = target_corr_abs.drop('good_client?')
target_corr = target_corr.drop('good_client?')

print("Strongest Correlations with Good Client (by absolute value):\n")
for feature, abs_corr in target_corr_abs.items():
    original_corr = target_corr[feature]
    correlation_direction = "positive" if original_corr > 0 else "negative"
    print(f"{feature:30s} | r = {original_corr:7.4f} | |r| = {abs_corr:.4f} ({correlation_direction})")


Based on the correlation analysis, we'll explore the relationships between the strongest features (`duration_months`, `credit_amount`, `age`, `installment_rate`):


In [ ]:
strongest_features = ['duration_months', 'credit_amount', 'age', 'installment_rate']

In [ ]:
# Scatterplot with class separation (using pairs of features)
viz_data = df_standard[strongest_features + ['good_client?']].copy()
viz_data['client_class'] = viz_data['good_client?'].map({1: 'Good', 0: 'Bad'})


fig, axes = plt.subplots(2, 3, figsize=(14, 10))
axes = axes.flatten()

feature_pairs = [
    (0, 1),  # duration_months vs credit_amount
    (0, 2),  # duration_months vs age
    (1, 2),  # credit_amount vs age
    (2, 3),  # age vs installment_rate
    (1, 3),  # credit_amount vs installment_rate
    (0, 3)   # duration_months vs installment_rate
]

for idx, (i, j) in enumerate(feature_pairs):
    feat1 = strongest_features[i]
    feat2 = strongest_features[j]
    
    for client_type in ['Good', 'Bad']:
        mask = viz_data['client_class'] == client_type
        color = 'green' if client_type == 'Good' else 'red'
        alpha = 0.6
        axes[idx].scatter(
            viz_data.loc[mask, feat1],
            viz_data.loc[mask, feat2],
            c=color,
            label=client_type,
            alpha=alpha,
            s=50,
            edgecolors='black',
            linewidth=0.5
        )
    
    axes[idx].set_xlabel(feat1.replace('_', ' ').title(), fontsize=11, weight='bold')
    axes[idx].set_ylabel(feat2.replace('_', ' ').title(), fontsize=11, weight='bold')
    axes[idx].set_title(f'{feat1} vs {feat2}', fontsize=12, weight='bold')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.suptitle('Scatterplot: Class Separation in Feature Space', fontsize=14, weight='bold', y=1.00)
plt.tight_layout()
plt.show()


In [ ]:
# Overlapping histograms for each feature
viz_data = df_standard[strongest_features + ['good_client?']].copy()
viz_data['client_class'] = viz_data['good_client?'].map({1: 'Good', 0: 'Bad'})

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.flatten()

for i, feat in enumerate(strongest_features):
    ax = axes[i]
    # Plot Good clients
    sns.histplot(
        viz_data[viz_data['client_class'] == 'Good'][feat],
        color='green', label='Good', kde=True, stat='count', element='step',
        fill=True, alpha=0.5, bins=30, ax=ax
    )
    # Plot Bad clients
    sns.histplot(
        viz_data[viz_data['client_class'] == 'Bad'][feat],
        color='red', label='Bad', kde=True, stat='count', element='step',
        fill=True, alpha=0.5, bins=30, ax=ax
    )

    ax.set_title(feat.replace('_', ' ').title(), fontsize=13, weight='bold')
    ax.set_xlabel(feat.replace('_', ' ').title())
    ax.set_ylabel('Count')
    ax.grid(axis='y', linestyle='--', alpha=0.2)
    ax.legend()

plt.suptitle('Overlapping Histograms of Strongest Features by Client Class', fontsize=16, weight='bold', y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

Based on the correlation values and the visual patterns in both the scatterplots and overlapping histograms (Images 1 and 2), we can observe that none of the main features has a strong individual relationship with the probability of being a good client. The highest absolute correlation is for duration_months (r = -0.21), indicating that clients with longer loan durations are slightly more likely to default. Similarly, higher credit_amount is weakly associated with higher default risk (r = -0.15). The age feature has a weak positive correlation (r = 0.09), suggesting that older clients are marginally more likely to be considered good clients, but the overlap between classes is substantial. Finally, installment_rate shows a very weak negative correlation.

The scatterplots confirm significant overlap between good and bad clients across all feature pairs, without clear boundaries for class separation. The overlapping histograms reinforce that although there are trends (for example, “bad” clients more frequently have longer durations and higher credit amounts), these patterns are not strong enough for a single feature to distinguish between classes. Overall, these results indicate that no single variable can reliably classify the clients, and effective models will likely require combining several attributes and possibly non-linear relationships.

## 3. Exporting Dataset

In [ ]:
file_out_path = '../data/processed'

df_standard.to_csv(file_out_path + '/german_df_processed_1.csv', index=False)